# Reviewer 1 Comment 5: unaugmented-source control
GPU runtime. Uses the existing Drive folder SROIE_Current_C2. Add genuine SROIE-original-test.zip to that folder. Other inputs are unchanged. All 350 source hashes must match before inference. Three conditions are rerun with a fixed model revision; no old predictions are mixed in. Checkpoint resumes with the same inputs/environment. This experiment isolates augmentation effects; it does not test masking task-relevant source fields or other document types. No results are claimed until execution.


In [ ]:
!pip -q install "transformers>=4.46,<5" sentencepiece accelerate pandas scipy
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import io,json,zipfile,re,hashlib
import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import DonutProcessor,VisionEncoderDecoderModel
BASE=Path('/content/drive/MyDrive/SROIE_Current_C2')
ORIGINAL=BASE/'SROIE-PII-full971-images-test.zip'
MASKED=BASE/'SROIE_Current_C2_Rendered350.zip'
METADATA=BASE/'SROIE-PII-full971-metadata.zip'
OUT=BASE/'Comment5_Results';OUT.mkdir(exist_ok=True)
assert torch.cuda.is_available(),'Select GPU runtime'
with zipfile.ZipFile(METADATA) as z:docs=[d for d in json.loads(z.read('sroie_pii_full971.json'))['documents'] if d['split']=='test']
assert len(docs)==350
MODEL='philschmid/donut-base-sroie'
REVISION='51e72e5d160efa472ee3958d79a766faa42f6c5d'
processor=DonutProcessor.from_pretrained(MODEL,revision=REVISION)
model=VisionEncoderDecoderModel.from_pretrained(MODEL,revision=REVISION).to('cuda').eval()
revision=getattr(model.config,'_commit_hash',None)
prompt=processor.tokenizer('<s>',add_special_tokens=False,return_tensors='pt').input_ids.to('cuda')
fields=['company','date','address','total']
def norm(s):return re.sub('[^a-z0-9]','',str(s or '').casefold())

SOURCE=BASE/'SROIE-original-test.zip'
assert SOURCE.exists(), 'Add the genuine unaugmented original image ZIP; do not rename the augmented ZIP.'

# Verify exact source provenance before inference. Filenames may be nested.
source_names={}
with zipfile.ZipFile(SOURCE) as z:
 for n in z.namelist():
  if n.lower().endswith(('.jpg','.jpeg','.png')):
   key=Path(n).stem
   if key in source_names:raise ValueError('Duplicate image ID: '+key)
   source_names[key]=n
 checks=[]
 for d in docs:
  key=d['document_id'];assert key in source_names, 'Missing original: '+key
  data=z.read(source_names[key]);sha=hashlib.sha256(data).hexdigest()
  assert sha==d['source_image_sha256'], 'Original hash mismatch: '+key
  im=Image.open(io.BytesIO(data));assert im.size==(d['width'],d['height'])
  checks.append({'document_id':key,'sha256':sha,'matches_source':True})
pd.DataFrame(checks).to_csv(OUT/'source_checks.csv',index=False)
# Record packages and archive fingerprints. Checkpoints tied to this exact run.
import importlib.metadata
run={'model':MODEL,'revision':REVISION,'documents':350,'conditions':['SOURCE_UNAUGMENTED','AUGMENTED_UNMASKED','MASKED_SHARED'],
'archive_sha256':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in [SOURCE,ORIGINAL,MASKED,METADATA]},
'versions':{k:importlib.metadata.version(k) for k in ['torch','transformers','numpy','pandas','pillow']}}
runpath=OUT/'run_spec.json'
if runpath.exists():assert json.loads(runpath.read_text())==run,'Inputs or environment changed: use a new output folder.'
else:runpath.write_text(json.dumps(run,indent=2))


In [ ]:
checkpoint=OUT/'checkpoint_predictions.csv'
records=pd.read_csv(checkpoint,keep_default_na=False).to_dict('records') if checkpoint.exists() else []
done={(r['document_id'],r['condition']) for r in records}
assert len(done)==len(records)
with zipfile.ZipFile(SOURCE) as zs,zipfile.ZipFile(ORIGINAL) as zo,zipfile.ZipFile(MASKED) as zm:
 for condition,z in [('SOURCE_UNAUGMENTED',zs),('AUGMENTED_UNMASKED',zo),('MASKED_SHARED',zm)]:
  for d in docs:
   if (d['document_id'],condition) in done:continue
   name=source_names[d['document_id']] if condition=='SOURCE_UNAUGMENTED' else (d['image_file'] if condition=='AUGMENTED_UNMASKED' else 'shared_Strong_RA_C2/'+d['document_id']+'.png')
   im=Image.open(io.BytesIO(z.read(name))).convert('RGB')
   pixels=processor(im,return_tensors='pt').pixel_values.to('cuda')
   with torch.inference_mode():
    seq=model.generate(pixels,decoder_input_ids=prompt,max_length=model.decoder.config.max_position_embeddings,early_stopping=True,pad_token_id=processor.tokenizer.pad_token_id,eos_token_id=processor.tokenizer.eos_token_id,use_cache=True,num_beams=1,bad_words_ids=[[processor.tokenizer.unk_token_id]])
   raw=processor.batch_decode(seq)[0]
   cleaned=raw.replace(processor.tokenizer.eos_token or '', '').replace(processor.tokenizer.pad_token or '', '').strip()
   try:pred=processor.token2json(cleaned)
   except Exception:pred={}
   if not isinstance(pred,dict):pred={}
   r={'document_id':d['document_id'],'condition':condition,'raw_prediction':raw}
   exact=[]
   for f in fields:
    gt=d['source_entities'][f];pv=pred.get(f,'')
    r['gt_'+f]=gt;r['pred_'+f]=str(pv);r['exact_'+f]=int(norm(gt)==norm(pv));exact.append(r['exact_'+f])
   r['field_exact']=float(np.mean(exact));records.append(r)
   pd.DataFrame(records).to_csv(OUT/'checkpoint_predictions.csv',index=False)
   if len(records)%25==0:print(len(records),'/1050')


In [ ]:

df=pd.DataFrame(records);assert len(df)==1050
assert not df.duplicated(['document_id','condition']).any()
df.to_csv(OUT/'predictions.csv',index=False)
summary=df.groupby('condition')['field_exact'].agg(['mean','count']);summary.to_csv(OUT/'summary.csv')
piv=df.pivot(index='document_id',columns='condition',values='field_exact').reindex([d['document_id'] for d in docs])
comparisons=[]
for a,b,label in [('SOURCE_UNAUGMENTED','AUGMENTED_UNMASKED','insertion_effect'),('AUGMENTED_UNMASKED','MASKED_SHARED','masking_effect'),('SOURCE_UNAUGMENTED','MASKED_SHARED','net_effect')]:
 delta=(piv[b]-piv[a]).to_numpy();rng=np.random.default_rng(20261003)
 boot=np.array([rng.choice(delta,len(delta),replace=True).mean() for _ in range(10000)])
 lo,hi=np.quantile(boot,[.025,.975])
 comparisons.append({'comparison':label,'reference':a,'evaluated':b,'difference':float(delta.mean()),'ci95_low':float(lo),'ci95_high':float(hi)})
pd.DataFrame(comparisons).to_csv(OUT/'comparisons.csv',index=False)
byfield=[]
for condition,g in df.groupby('condition'):
 for f in fields:byfield.append({'condition':condition,'field':f,'correct':int(g['exact_'+f].sum()),'documents':len(g),'accuracy':float(g['exact_'+f].mean())})
pd.DataFrame(byfield).to_csv(OUT/'field_results.csv',index=False)
print(summary);print(pd.DataFrame(comparisons))
print('Finished. Return summary.csv, comparisons.csv, field_results.csv, predictions.csv, source_checks.csv and run_spec.json.')
